In [13]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# Modelos baseline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# ============================================================
# 1. Cargar dataset procesado
# ============================================================
df = pd.read_csv("../data/processed/04_default_credit_features.csv")
df = df.drop(columns=["ID"])

TARGET = "default payment next month"
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Detectar columnas
cat_cols = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

# ============================================================
# 2. Construir preprocesador (NO cargarlo desde archivo)
# ============================================================
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preproc = ColumnTransformer([
    ("num", numeric_pipeline, num_cols),
    ("cat", categorical_pipeline, cat_cols)
])

# ============================================================
# 3. Train/Test Split
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ============================================================
# 4. Modelos baseline
# ============================================================
modelos = {
    "LogisticRegression": LogisticRegression(max_iter=5000),
    "DecisionTree": DecisionTreeClassifier(),
    "RandomForest": RandomForestClassifier(),
    "SVM": SVC(probability=True),
    "KNN": KNeighborsClassifier(),
}

# ============================================================
# 5. Entrenar y guardar modelos
# ============================================================
for nombre, modelo in modelos.items():
    print(f"\nEntrenando {nombre}...")

    pipe = Pipeline([
        ("preprocessing", preproc),
        ("clf", modelo)
    ])

    pipe.fit(X_train, y_train)

    joblib.dump(pipe, f"../models/{nombre}.joblib")
    print(f"✓ {nombre} guardado en ../models/{nombre}.joblib")


Entrenando LogisticRegression...
✓ LogisticRegression guardado en ../models/LogisticRegression.joblib

Entrenando DecisionTree...
✓ DecisionTree guardado en ../models/DecisionTree.joblib

Entrenando RandomForest...
✓ RandomForest guardado en ../models/RandomForest.joblib

Entrenando SVM...
✓ SVM guardado en ../models/SVM.joblib

Entrenando KNN...
✓ KNN guardado en ../models/KNN.joblib




1. Logistic Regression
----------------------
Qué es:
    - Un modelo lineal para clasificación binaria.

Cómo funciona:
    - Calcula una combinación lineal de las variables y aplica
      una función sigmoide para obtener probabilidades entre 0 y 1.

Por qué se incluye:
    - Es rápido, interpretable y sirve como baseline inicial.
    - Permite entender relaciones lineales entre variables.

-----------------------------------------------------------

2. Decision Tree
----------------
Qué es:
    - Un modelo basado en reglas tipo árbol de decisión.

Cómo funciona:
    - Divide los datos en nodos según la característica que mejor
      separa las clases (criterio Gini o Entropía).
    - Cada hoja representa una predicción.

Por qué se incluye:
    - Fácil de interpretar.
    - Captura relaciones no lineales.
    - Permite identificar variables importantes.

-----------------------------------------------------------

3. Random Forest
----------------
Qué es:
    - Un modelo de ensamble compuesto por múltiples árboles de decisión.

Cómo funciona:
    - Entrena muchos árboles usando subconjuntos aleatorios de datos
      y características (bagging).
    - La predicción final es el voto mayoritario de todos los árboles.

Por qué se incluye:
    - Reduce el overfitting de un solo árbol.
    - Suele ofrecer muy buen rendimiento sin mucha configuración.
    - Maneja bien datos ruidosos y relaciones complejas.

-----------------------------------------------------------

4. Support Vector Machine (SVM)
-------------------------------
Qué es:
    - Un modelo que busca el hiperplano óptimo que separa las clases.

Cómo funciona:
    - Maximiza el margen entre clases.
    - Puede usar kernels para capturar relaciones no lineales.
    - Se usa con probability=True para obtener probabilidades.

Por qué se incluye:
    - Funciona bien en espacios de alta dimensión.
    - Es robusto ante outliers.
    - Es un baseline fuerte para clasificación.

-----------------------------------------------------------

5. K-Nearest Neighbors (KNN)
----------------------------
Qué es:
    - Un modelo basado en la similitud entre observaciones.

Cómo funciona:
    - Para clasificar un punto nuevo:
        1. Calcula la distancia a todos los puntos del entrenamiento.
        2. Selecciona los k más cercanos.
        3. Predice la clase más frecuente entre ellos.

Por qué se incluye:
    - Es simple y no paramétrico.
    - Sirve como baseline para comparar contra modelos más complejos.
    - Captura patrones locales en los datos.

